# Selección del modelo de predicción

El objetivo de este notebook es comparar distintos algoritmos de regresión para identificar cuál ofrece el mejor desempeño en la predicción del precio promedio semanal de productos agrícolas.

Cada modelo es evaluado utilizando validación cruzada sobre el conjunto de entrenamiento, permitiendo comparar su capacidad predictiva mediante métricas consistentes y minimizar el riesgo de sobreajuste durante la selección.

El modelo seleccionado será posteriormente entrenado y evaluado en el notebook de testing, donde se realizará la evaluación final sobre el conjunto de prueba y se exportará el pipeline para su uso en inferencia.


## Modelos evaluados

Con el objetivo de seleccionar el algoritmo más adecuado para el problema de predicción, se compararán los siguientes modelos de regresión:

- **Regresión Lineal:** utilizada como modelo base (baseline) para establecer una referencia de desempeño.
- **Árbol de Decisión:** modelo no lineal capaz de capturar relaciones complejas entre las variables. Al tener otros modelos basados en arboles ofrece un buen punto de referencia.
- **Random Forest:** conjunto de múltiples árboles de decisión que reduce la varianza y mejora la capacidad de generalización.
- **Gradient Boosting Regressor:** modelo de ensamble basado en boosting que construye árboles de forma secuencial para minimizar el error.
- **HistGradientBoosting Regressor:** variante optimizada de Gradient Boosting que discretiza las variables continuas en histogramas, permitiendo un entrenamiento significativamente más rápido sobre conjuntos de datos grandes.

Todos los modelos serán evaluados mediante validación cruzada utilizando las mismas métricas de desempeño.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
    FunctionTransformer
)

from sklearn.model_selection import (
    train_test_split,
    cross_validate,
    KFold
)

#modelos
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import HistGradientBoostingRegressor

#modelos más simples con los que comparar
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor


In [2]:
df = pd.read_csv("./Data/processed_agricultural_prices.csv")

In [4]:
df.dtypes

semana                    int64
fecha_inicio                str
region                      str
tipo_punto_monitoreo        str
producto                    str
variedad                    str
calidad                     str
unidad                      str
precio_minimo           float64
precio_maximo           float64
precio_promedio         float64
anio                      int64
mes                       int64
producto_variedad           str
temperature_2m_mean     float64
temperature_2m_max      float64
temperature_2m_min      float64
precipitation_sum       float64
dtype: object

In [5]:
# Variables que describen la variable objetivo
#o que no entregan informacion (proucto y variedad son un solo campo ahora, unidad solo presenta un valor $/kg)
#fecha inicio fue deconstruida en 3 campos : semana,mes,anio
#mes y semana tenian 0.96 de correlación... dejaré semana ya que ese es el formato en el que se obtuvo la información
df.drop(
    columns=[
        "precio_minimo",
        "precio_maximo",
        "producto",
        "variedad",
        "unidad",
        "fecha_inicio",
        "mes"
    ],
    inplace=True
)

### Feature Engineering

- Antes de crear el train test split me interesa crear columnas con datos deterministas como lo serían el lag de algunas variables.
- tomar en cuenta el precio o condiciones climaticas de la semana anterior. Si este nuevo campo resulta en ganancia de información: considerar la creación de una columna(as) con otros rangos de lag.
- luego durante la validación cruzada se determinará si las columnas creadas ofrecieron valor, en caso contrario serán eliminadas.


In [6]:
df = df.sort_values([
    "producto_variedad",
    "region",
    "tipo_punto_monitoreo",
    "calidad",
    "anio",
    "semana"
])

In [7]:
# Lag: precio_promedio de la semana anterior
df["precio_lag_1"] = (
    df.groupby([
        "producto_variedad",
        "region",
        "tipo_punto_monitoreo",
        "calidad"
    ])["precio_promedio"]
    .shift(1)
)

In [8]:
#la primera fila del dataset tendrá un NAN al no tene runa semana anteior a ella asi que la borraremos
df = df.dropna(subset=["precio_lag_1"])

In [9]:
df.head()

,semana,region,tipo_punto_monitoreo,calidad,precio_promedio,anio,producto_variedad,temperature_2m_mean,temperature_2m_max,temperature_2m_min,precipitation_sum,precio_lag_1
371,2,región_de_arica_y_parinacota,mercado_minorista,extra,3750.0,2020,aceituna_amarga,23.471429,27.142857,20.585714,0.028571,3750.0
849,3,región_de_arica_y_parinacota,mercado_minorista,extra,3667.0,2020,aceituna_amarga,23.671429,26.628571,21.314286,0.242857,3750.0
1328,4,región_de_arica_y_parinacota,mercado_minorista,extra,3750.0,2020,aceituna_amarga,24.957143,27.814286,22.885714,0.914286,3667.0
1801,5,región_de_arica_y_parinacota,mercado_minorista,extra,3700.0,2020,aceituna_amarga,25.071429,28.257143,22.485714,0.214286,3750.0
2387,6,región_de_arica_y_parinacota,mercado_minorista,extra,3667.0,2020,aceituna_amarga,23.657143,27.185714,20.871429,0.014286,3700.0


In [12]:
# Agregar al dataset, sin esto el siguiente notebook podría mandar errores; o peor: no dar error y no darme cuenta que no usé la variable nueva
df.to_csv("./Data/ModelReady/fruit_prices_clean.csv", index=False)

## División de entrenamiento y prueba de modelos

El conjunto de datos se divide en entrenamiento y prueba.

El conjunto de entrenamiento será utilizado durante la validación cruzada para seleccionar el mejor modelo, mientras que el conjunto de prueba permanecerá completamente aislado hasta la evaluación final.

In [30]:
categorical_cols = [
    "producto_variedad",
    "calidad",
    "region",
    "tipo_punto_monitoreo",
]
# a este tengo que aplicarle log +1
log_cols = [
    "precipitation_sum"
]

numerical_cols = [
    "semana",
    "anio",
    "temperature_2m_mean",
    "temperature_2m_max",
    "temperature_2m_min",
    "precio_lag_1"
]

scaled_numerical_cols = [
    c for c in numerical_cols
    if c not in log_cols
]

In [31]:
X = df[numerical_cols + categorical_cols + log_cols]
y = df["precio_promedio"]


In [32]:
# 80% train , 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

## Preprocesamiento

Las variables categóricas son transformadas mediante One Hot Encoding mientras que las númericas son escaladas y se corrigen las colas vistas en el documento anterior mediante log()

Este proceso se integra dentro de un Pipeline para evitar fuga de información durante la validación cruzada.

In [33]:
#Aplicar log +1 a precipitacion_sum
# Nota: necesito aplicar log y normalizar al mismo tiempo si no quiero que alguno de estos se omita o se creen 2 columnas derivadas al hacerlo por separado
log_numeric = Pipeline([
    ("log", FunctionTransformer(np.log1p)),
    ("scaler", StandardScaler())
])

preprocessor = ColumnTransformer(
    transformers=[
        (
            "log",
            log_numeric,
            log_cols
        ),
        (
            "numeric",
            StandardScaler(),
            scaled_numerical_cols
        ),
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_cols
        )
    ]
)

## Modelos evaluados

Se compararán distintos algoritmos de regresión con el objetivo de identificar cuál ofrece los mejores resultados.

Nota: Los modelos serán evaluados uno por uno en detrimento del orden y visiblidad para limitar el tiempo de ejecución y uso de recursos en mi equipo personal.

In [55]:
rf = RandomForestRegressor(
    random_state=42,
    n_estimators=25,
    n_jobs=-1
)


gbr = GradientBoostingRegressor(
    random_state=42,
    n_estimators=50
)

hist_gbr = HistGradientBoostingRegressor(
    random_state=42,
    max_iter=50
)



#Estos son modelos más simples pero quiero observar la diferencia respecto a los 3 anteriores y me parece que generan una buena comparación.
lr = LinearRegression()

dt = DecisionTreeRegressor(
    random_state=42
)
# Definir que otros odelos usar

In [35]:
pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", rf)
])

#repetir para todos los modelos.

### Revisar cuantas columnas deja el One Hot Encoding

In [22]:
X_trans = preprocessor.fit_transform(X_train)
print(X_trans.shape)
#107

(125131, 107)


### Prueba rapida de demora para un solo fit

In [23]:
import time

inicio = time.time()
pipeline.fit(X_train, y_train)
print(time.time() - inicio)
#cuando estimadores = 100

609.1546633243561


## Validación cruzada

Se utiliza validación cruzada de cinco particiones (5-Fold Cross Validation) para obtener una estimación más robusta del rendimiento de cada modelo.

En cada iteración un subconjunto distinto actúa como validación mientras el resto se utiliza para entrenamiento.

Las métricas reportadas corresponden al promedio obtenido durante las cinco iteraciones.

In [52]:
models_scores = []

In [46]:
cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

### Random Forest

In [47]:
inicio = time.time()

scores_rf = cross_validate(
    pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring=[
        "r2",
        "neg_mean_absolute_error",
        "neg_root_mean_squared_error"
    ],
    return_train_score=True
)
print(f"Tiempo para Random Forest: {time.time() - inicio:.1f} s")
#repetir para todos los modelos

Tiempo para Random Forest: 549.3 s


In [49]:
#revisar scores para todos los modelos y definir el mejor
results = pd.DataFrame(scores_rf)

results

,fit_time,score_time,test_r2,train_r2,test_neg_mean_absolute_error,train_neg_mean_absolute_error,test_neg_root_mean_squared_error,train_neg_root_mean_squared_error
0,110.201158,0.090216,0.945906,0.990081,-182.278023,-71.410533,-397.557537,-176.084670
1,109.782390,0.089573,0.936034,0.990982,-185.482695,-70.874797,-447.274145,-166.510581
2,109.683272,0.089183,0.943119,0.990781,-185.192958,-70.811928,-423.693135,-168.164994
3,108.388546,0.089222,0.940581,0.991235,-186.308236,-70.703597,-430.127795,-164.258420
4,109.220265,0.088891,0.931673,0.990998,-187.778002,-70.910850,-460.700538,-166.511542


In [50]:
summary = pd.DataFrame({
    "R2 promedio": [scores_rf["test_r2"].mean()],
    "MAE promedio": [-scores_rf["test_neg_mean_absolute_error"].mean()],
    "RMSE promedio": [-scores_rf["test_neg_root_mean_squared_error"].mean()],
    "Tiempo entrenamiento promedio": [scores_rf["fit_time"].mean()]
})

summary

,R2 promedio,MAE promedio,RMSE promedio,Tiempo entrenamiento promedio
0,0.939463,185.407983,431.87063,109.455126


In [53]:
models_scores.append({
    "Modelo": "Random Forest",
    "R2": scores_rf["test_r2"].mean(),
    "MAE": -scores_rf["test_neg_mean_absolute_error"].mean(),
    "RMSE": -scores_rf["test_neg_root_mean_squared_error"].mean()
})

### Gradient Boosting

In [57]:
pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", gbr)
])

In [58]:
inicio = time.time()

scores_gbr = cross_validate(
    pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring=[
        "r2",
        "neg_mean_absolute_error",
        "neg_root_mean_squared_error"
    ],
    return_train_score=True
)
print(f"Tiempo para Gradient Boosting: {time.time() - inicio:.1f} s")

Tiempo para Random Forest: 41.3 s


In [59]:
results = pd.DataFrame(scores_rf)

results

,fit_time,score_time,test_r2,train_r2,test_neg_mean_absolute_error,train_neg_mean_absolute_error,test_neg_root_mean_squared_error,train_neg_root_mean_squared_error
0,110.201158,0.090216,0.945906,0.990081,-182.278023,-71.410533,-397.557537,-176.084670
1,109.782390,0.089573,0.936034,0.990982,-185.482695,-70.874797,-447.274145,-166.510581
2,109.683272,0.089183,0.943119,0.990781,-185.192958,-70.811928,-423.693135,-168.164994
3,108.388546,0.089222,0.940581,0.991235,-186.308236,-70.703597,-430.127795,-164.258420
4,109.220265,0.088891,0.931673,0.990998,-187.778002,-70.910850,-460.700538,-166.511542


In [60]:
models_scores.append({
    "Modelo": "Gradient Boosting",
    "R2": scores_gbr["test_r2"].mean(),
    "MAE": -scores_gbr["test_neg_mean_absolute_error"].mean(),
    "RMSE": -scores_gbr["test_neg_root_mean_squared_error"].mean()
})

### Hist Gradient Boosting

#### Observación: no todos los modelos aceptan el mismo tipo de preprocesamiento

Para los modelos anteriores se utilizó:

`OneHotEncoder(handle_unknown="ignore")`

Esto genera una matriz de datos **sparse**, donde los `0` (y One Hot Encoder genera muchos `0`) se "ignoran"; se guardan únicamente los valores `1` y sus posiciones, haciendo la representación más ligera.

El modelo **Hist Gradient Boosting** obliga a utilizar una matriz de datos **dense**, donde se almacena toda la información, incluyendo los valores `0`.

Esto resultará en una mayor cantidad de datos almacenados, por lo que el tiempo de entrenamiento y consumo de memoria podrían verse afectados.

In [67]:
log_numeric = Pipeline([
    ("log", FunctionTransformer(np.log1p)),
    ("scaler", StandardScaler())
])

preprocessor_dense = ColumnTransformer(
    transformers=[
        (
            "log",
            log_numeric,
            log_cols
        ),
        (
            "numeric",
            StandardScaler(),
            scaled_numerical_cols
        ),
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_cols
        )
    ]
)

In [68]:
pipeline = Pipeline([
    ("preprocessing", preprocessor_dense),
    ("model", hist_gbr)
])

In [69]:
inicio = time.time()

scores_hist_gbr = cross_validate(
    pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring=[
        "r2",
        "neg_mean_absolute_error",
        "neg_root_mean_squared_error"
    ],
    return_train_score=True
)
print(f"Tiempo para Hist Gradient Boosting: {time.time() - inicio:.1f} s")

Tiempo para Hist Gradient Boosting: 9.9 s


In [70]:
results = pd.DataFrame(scores_rf)

results

,fit_time,score_time,test_r2,train_r2,test_neg_mean_absolute_error,train_neg_mean_absolute_error,test_neg_root_mean_squared_error,train_neg_root_mean_squared_error
0,110.201158,0.090216,0.945906,0.990081,-182.278023,-71.410533,-397.557537,-176.084670
1,109.782390,0.089573,0.936034,0.990982,-185.482695,-70.874797,-447.274145,-166.510581
2,109.683272,0.089183,0.943119,0.990781,-185.192958,-70.811928,-423.693135,-168.164994
3,108.388546,0.089222,0.940581,0.991235,-186.308236,-70.703597,-430.127795,-164.258420
4,109.220265,0.088891,0.931673,0.990998,-187.778002,-70.910850,-460.700538,-166.511542


In [71]:
models_scores.append({
    "Modelo": "Hist Gradient Boosting",
    "R2": scores_hist_gbr["test_r2"].mean(),
    "MAE": -scores_hist_gbr["test_neg_mean_absolute_error"].mean(),
    "RMSE": -scores_hist_gbr["test_neg_root_mean_squared_error"].mean()
})

### Linear Regression

In [74]:
pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", lr)
])

In [77]:
inicio = time.time()

scores_lr = cross_validate(
    pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring=[
        "r2",
        "neg_mean_absolute_error",
        "neg_root_mean_squared_error"
    ],
    return_train_score=True
)
print(f"Tiempo para Linear Regression: {time.time() - inicio:.1f} s")

Tiempo para Linear Regression: 3.0 s


In [76]:
results = pd.DataFrame(scores_lr)

results

,fit_time,score_time,test_r2,train_r2,test_neg_mean_absolute_error,train_neg_mean_absolute_error,test_neg_root_mean_squared_error,train_neg_root_mean_squared_error
0,110.201158,0.090216,0.945906,0.990081,-182.278023,-71.410533,-397.557537,-176.084670
1,109.782390,0.089573,0.936034,0.990982,-185.482695,-70.874797,-447.274145,-166.510581
2,109.683272,0.089183,0.943119,0.990781,-185.192958,-70.811928,-423.693135,-168.164994
3,108.388546,0.089222,0.940581,0.991235,-186.308236,-70.703597,-430.127795,-164.258420
4,109.220265,0.088891,0.931673,0.990998,-187.778002,-70.910850,-460.700538,-166.511542


In [87]:
models_scores.append({
    "Modelo": "Linear Regression",
    "R2": scores_lr["test_r2"].mean(),
    "MAE": -scores_lr["test_neg_mean_absolute_error"].mean(),
    "RMSE": -scores_lr["test_neg_root_mean_squared_error"].mean()
})

In [88]:
models_scores

[{'Modelo': 'Decision Tree',
  'R2': np.float64(0.9394629183152456),
  'MAE': np.float64(185.4079827537526),
  'RMSE': np.float64(431.87062994294513)},
 {'Modelo': 'Gradient Boosting',
  'R2': np.float64(0.9364259606029839),
  'MAE': np.float64(193.63630382830672),
  'RMSE': np.float64(442.54911404043105)},
 {'Modelo': 'Hist Gradient Boosting',
  'R2': np.float64(0.9397648634001918),
  'MAE': np.float64(189.82633727221176),
  'RMSE': np.float64(430.9404196497123)},
 {'Modelo': 'Linear Regression',
  'R2': np.float64(0.9348744247724777),
  'MAE': np.float64(193.01475467832321),
  'RMSE': np.float64(447.88917824082284)}]

### Desicion Tree

In [84]:
pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", dt)
])

In [85]:
inicio = time.time()

scores_dt = cross_validate(
    pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring=[
        "r2",
        "neg_mean_absolute_error",
        "neg_root_mean_squared_error"
    ],
    return_train_score=True
)
print(f"Tiempo para Desicion Tree: {time.time() - inicio:.1f} s")

Tiempo para Desicion Tree: 186.5 s


In [89]:
results = pd.DataFrame(scores_dt)

results

,fit_time,score_time,test_r2,train_r2,test_neg_mean_absolute_error,train_neg_mean_absolute_error,test_neg_root_mean_squared_error,train_neg_root_mean_squared_error
0,37.747947,0.040854,0.902387,1.0,-240.481720,-0.0,-534.049228,-0.0
1,37.660888,0.039894,0.894148,1.0,-246.824702,-0.0,-575.372626,-0.0
2,37.021073,0.039590,0.897384,1.0,-246.932630,-0.0,-569.084000,-0.0
3,37.012394,0.039246,0.896288,1.0,-247.483657,-0.0,-568.264043,-0.0
4,36.145442,0.040090,0.889370,1.0,-247.453249,-0.0,-586.220411,-0.0


In [90]:
models_scores.append({
    "Modelo": "Desicion Tree",
    "R2": scores_dt["test_r2"].mean(),
    "MAE": -scores_dt["test_neg_mean_absolute_error"].mean(),
    "RMSE": -scores_dt["test_neg_root_mean_squared_error"].mean()
})

In [92]:
results_df = pd.DataFrame(models_scores)
results_df

,Modelo,R2,MAE,RMSE
0,Decision Tree,0.939463,185.407983,431.870630
1,Gradient Boosting,0.936426,193.636304,442.549114
2,Hist Gradient Boosting,0.939765,189.826337,430.940420
3,Linear Regression,0.934874,193.014755,447.889178
4,Desicion Tree,0.895915,245.835192,566.598062


### Conclusión de la comparación de modelos

Tras evaluar los distintos modelos mediante validación cruzada de 5 particiones (`KFold`), se observa que los modelos basados en ensamblados de árboles obtienen el mejor desempeño en la predicción del precio promedio.

| Modelo | R² | MAE | RMSE |
|:--|--:|--:|--:|
| Hist Gradient Boosting | **0.9398** | 189.83 | **430.94** |
| Random Forest | 0.9395 | **185.41** | 431.87 |
| Gradient Boosting | 0.9364 | 193.64 | 442.55 |
| Linear Regression | 0.9349 | 193.01 | 447.89 |
| Decision Tree | 0.8959 | 245.84 | 566.60 |

Aunque **Random Forest** obtuvo el menor **MAE**, las diferencias con **Hist Gradient Boosting** son mínimas. Por otro lado, **Hist Gradient Boosting** alcanzó el mejor **R²** y el menor **RMSE**, métricas que indican una mejor capacidad de generalización y un menor impacto de los errores grandes.

Un aspecto determinante fue el tiempo de entrenamiento. Mientras que **Random Forest** requirió aproximadamente **9 minutos** para completar la validación cruzada, **Hist Gradient Boosting** finalizó el mismo proceso en **cerca de 10 segundos**, obteniendo un rendimiento prácticamente equivalente.

Considerando tanto el desempeño predictivo como la eficiencia computacional, **Hist Gradient Boosting** fue seleccionado como el modelo final para este proyecto.

## Modelo seleccionado

Tras comparar todos los algoritmos, HistGradientBoosting fue seleccionado para la evaluación final debido a su excelente rendimiento y bajo tiempo de entrenamiento.

A continuación se exporta el pipeline completo para reutilizarlo posteriormente durante la etapa de inferencia.

In [94]:
import joblib

In [95]:
pipeline = Pipeline([
    ("preprocessing", preprocessor_dense),
    ("model", hist_gbr)
])

In [96]:
joblib.dump(pipeline, "hist_gradient_boosting.pkl")

['hist_gradient_boosting.pkl']